## Interpret a Threedx Model and its Forecasts

Everything about a Threedx model is interpretable, because

- its forecast is a linear combination of past observations,
- you chose the loss function the model was fitted to minimize,
- and the trialled parameter combinations are deterministic.

Let's take a closer look at each of those aspects by fitting the model you've seen in the [Get Started](../howto/getstarted.ipynb) section and then inspecting it.

In [ ]:
import numpy as np
import threedx as tdx
import matplotlib.pyplot as plt

np.set_printoptions(precision=3)
rng = np.random.default_rng(seed=512)

y = rng.poisson(
    lam=np.maximum(
        0.1,
        1 + 10 * np.sin(2 * np.pi * np.arange(6, 59) / 12)
    ),
    size=53
)

In [ ]:
grid = tdx.initialize_parameters_at_random(
    size=2500,
    seed=729,
    include_edge_cases=True
)

model = tdx.Threedx(period_length=12, parameter_grid=grid)
model = model.fit(y=y, loss=tdx.mae)

sample_paths_from_observations = model.predict(
    horizon=12,
    # Using a very large number of samples to reduce the sampling error when
    # trying to reproduce the point prediction of the linear combination.
    # Not recommended in practice.
    n_samples=100_001,
    observation_driven=True,
    draw=None,
    seed=388
)

### A Threedx Forecast as Linear Combination

#### One-Step-Ahead Prediction

The one-step-ahead forecast that `Threedx.predict()` generates is a linear combination of the past observations `y` and the weights assigned to them by the fitted model. The weights of a fitted model can be accessed via the `.weights()` method. They are aligned with the past observations `y`, so that the linear combination or dot-product of weights and `y` will equal the one-step-ahead point prediction.

In [ ]:
print("one-step-ahead point prediction as linear combination of past observations:")
print(round(np.dot(model.weights(), y), 3))

print("")

print("one-step-ahead mean of sample paths as returned by .predict():")
print(round(np.mean(sample_paths_from_observations, axis=0)[0], 3))

# A small difference between the two values is unavoidable due to the sampling
# error introduced when drawing sample paths and then aggregating them.

Given that the forecast is a linear combination of past observations and the model's weights, it's fair to make statements such as the following:

In [ ]:
print(f"""
    The most recent observation contributes {round(model.weights()[-1]*100, 2)}%
    to the one-step-ahead prediction.
""")

print(f"""
    The observations of the most recent 12 months contribute
    {round(np.sum(model.weights()[(y.size-12):])*100, 2)}% to the one-step-ahead
    prediction.
""")

print(f"""
    If the observation a year ago had been 20 units larger,
    the one-step-ahead prediction would increase by
    {round(model.weights()[y.size-12]*20, 2)} units.
""")

#### k-Step-Ahead Prediction

The same reasoning can be extended to all k-step-ahead predictions with the caveat that Threedx forecasts are recursive in that the two-step-ahead prediction is the linear combination of the past observations *and* the one-step-ahead prediction, and so on.

Accordingly, the weights expand during prediction to align with both past observations and previously created predictions. Those weights are *not* returned by `.weights()` but can be derived from the model's parameters.

In [ ]:
weights_for_one_step_ahead_prediction = model.weights()

weights_for_two_step_ahead_prediction = tdx.weights_threedx(
    alpha=model.best_alpha,
    alpha_seasonal=model.best_alpha_seasonal,
    alpha_seasonal_decay=model.best_alpha_seasonal_decay,
    # The `n` argument is the only difference between the one-step-ahead and the
    # two-step-ahead prediction. As the weights need to align with more
    # observations (as they now include predictions, too), `n` increases.
    n=y.size+1,
    period_length=12,
)

Depending on the fitted model and the number of past observations, weights don't change by a lot as you expand them. In the case here, as little weight is assigned to the first observation, the weights barely change as they are expanded but rather shifted by one position. The most recent weight of 0.3259 will not be assigned to the most recent observation but to the first prediction.

In [ ]:
print(np.concatenate(
    [np.zeros(shape=(1,)), weights_for_one_step_ahead_prediction]
))

print(weights_for_two_step_ahead_prediction)

You can now recompute the linear combination using the expanded set of weights to derive the two-step-ahead prediction. First calculate the one-step-ahead prediction and concatenate it with past observations. Then use the extended history in a second dot product to arrive at the two-step-ahead prediction.

Comparing the two-step-ahead prediction against the mean of the two-step-ahead sample paths, results match closely.

In [ ]:
one_step_ahead_prediction = np.zeros((1,))
one_step_ahead_prediction[0] = np.dot(model.weights(), y)

print("two-step-ahead point prediction as linear combination of past observations:")
print(round(
    np.dot(
        weights_for_two_step_ahead_prediction,
        np.concatenate([y, one_step_ahead_prediction])
    ),
3))

print("")

print("two-step-ahead mean of sample paths as returned by .predict():")
print(round(np.mean(sample_paths_from_observations, axis=0)[1], 3))

# A small difference between the two values is unavoidable due to the sampling
# error introduced when drawing sample paths and then aggregating them.

### Visualize the Weights

It can be difficult to interpret the weights when the values are printed as above. Consider visualizing the weights using a barplot or similar methods. That makes it easy to see which past observations are weighted heavily by the model.

The plot below highlights that the seasonal average component dominates the model fitted above. Every prediction is primarily an average of past observations at the same point in the seasonal period. Weights are tilted slightly in favor of more recent observations.

In [ ]:
plt.bar(
    x=np.arange(0, weights_for_one_step_ahead_prediction.size, step=1),
    height=weights_for_one_step_ahead_prediction,
)

You should expect these weigths due to the large `alpha_seasonal` parameter of the fitted model (leading to the seasonal peaks) combined with a very small but not zero `alpha` parameter and a small `alpha_seasonal_decay` parameter, both contributing to decreasing weights for older observations.

In [ ]:
print(f"{model.best_alpha=}")
print(f"{model.best_alpha_seasonal=}")
print(f"{model.best_alpha_seasonal_decay=}")

### Parameter Combinations and Loss Function

It is unusual that Threedx does not use the loss function's gradients to fit the model's parameters. But with only three parameters, all constrained to a 0-to-1 range, to fit, it is not unreasonable to simply evaluate a grid of parameter combinations and pick the one that worked best (i.e., minimized the loss function). Yes, there is a good chance that there exists another parameter combination that would achieve a slightly improved training loss. But both parameter values and the resulting predictions are unlikely to differ much as long as a fairly exhaustive set of parameter combinations was trialled.

A similar idea was used in ["A white-boxed ISSM approach to estimate uncertainty distributions of Walmart sales" (2021)](https://arxiv.org/abs/2111.14721).

This provides an advantage in that it allows us to state with certainty that, within the set of trialled parameter combinations, we did indeed find the set of parameters that minimizes the loss function. In contrast, with other approaches of optimization, there is a chance of ending up in a weird local minimum without realizing it.

You can inspect the minimal training loss achieved by the fitted parameters:

In [ ]:
print(f"{model.minimal_loss=}")

Or you could print all losses of all trialled parameter combinations:

In [ ]:
print(model.losses)

This loss vector is aligned with the trialled parameter combinations, so you could, for example, print the first six parameter combinations and their corresponding losses:

In [ ]:
print(grid[0:6,])
print(model.losses[0:6])

Which shows that from the five edge cases, the third (seasonal average) performs best, while the fifth (average of last 12 observations) performs worst. The fourth (seasonal naive) comes closest to the third.

Additionally, the minimal training loss printed above (1.444) is not that much better than what the seasonal average achieves (1.518). In fact, given how the example series was created, the seasonal average is the correct model, but the observation noise leads to slightly different parameters---though still dominated by the seasonal average component, as you saw above.